<a href="https://colab.research.google.com/github/simaaronmaaan-ai/VOX/blob/main/uzbek_tts_dataset_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from huggingface_hub import hf_hub_download
import os

print("📥 Скачиваем USC (~9.5 GB)...")
archive_path = hf_hub_download(
    repo_id="issai/Uzbek_Speech_Corpus",
    filename="ISSAI_USC.tar.gz",
    repo_type="dataset",
    local_dir="/content/usc_raw"
)
print(f"✅ Скачан: {archive_path}")

📥 Скачиваем USC (~9.5 GB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ISSAI_USC.tar.gz:   0%|          | 0.00/9.47G [00:00<?, ?B/s]

✅ Скачан: /content/usc_raw/ISSAI_USC.tar.gz


In [ ]:
import tarfile, os
from pathlib import Path

EXTRACT_DIR = Path("/content/usc_extracted")
EXTRACT_DIR.mkdir(exist_ok=True)

print("📦 Распаковываем...")
with tarfile.open(archive_path, "r:gz") as tar:
    tar.extractall(EXTRACT_DIR)

# Смотрим структуру
for p in sorted(EXTRACT_DIR.rglob("*"))[:30]:
    print(p.relative_to(EXTRACT_DIR))

📦 Распаковываем...


/tmp/ipykernel_1164/3772193947.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(EXTRACT_DIR)


ISSAI_USC
ISSAI_USC/dev
ISSAI_USC/dev/1032229503_2_14073_1.txt
ISSAI_USC/dev/1032229503_2_14073_1.wav
ISSAI_USC/dev/1032229503_2_14075_2.txt
ISSAI_USC/dev/1032229503_2_14075_2.wav
ISSAI_USC/dev/1032229503_2_14081_1.txt
ISSAI_USC/dev/1032229503_2_14081_1.wav
ISSAI_USC/dev/1032229503_2_14083_1.txt
ISSAI_USC/dev/1032229503_2_14083_1.wav
ISSAI_USC/dev/1032229503_2_15885_2.txt
ISSAI_USC/dev/1032229503_2_15885_2.wav
ISSAI_USC/dev/1032229503_2_15886_2.txt
ISSAI_USC/dev/1032229503_2_15886_2.wav
ISSAI_USC/dev/1032229503_2_15893_2.txt
ISSAI_USC/dev/1032229503_2_15893_2.wav
ISSAI_USC/dev/1032229503_2_15895_2.txt
ISSAI_USC/dev/1032229503_2_15895_2.wav
ISSAI_USC/dev/1032229503_2_15896_1.txt
ISSAI_USC/dev/1032229503_2_15896_1.wav
ISSAI_USC/dev/1032229503_2_15899_1.txt
ISSAI_USC/dev/1032229503_2_15899_1.wav
ISSAI_USC/dev/1032229503_2_15901_1.txt
ISSAI_USC/dev/1032229503_2_15901_1.wav
ISSAI_USC/dev/1032229503_2_15905_2.txt
ISSAI_USC/dev/1032229503_2_15905_2.wav
ISSAI_USC/dev/1032229503_2_15906_2.txt
I

In [ ]:
import soundfile as sf
import numpy as np
from datasets import Dataset, Audio
from tqdm import tqdm

# USC хранит пары: audio.wav + audio.txt рядом
samples = []
wav_files = sorted(EXTRACT_DIR.rglob("*.wav"))
print(f"Найдено wav файлов: {len(wav_files)}")

for wav_path in tqdm(wav_files, desc="Парсим"):
    txt_path = wav_path.with_suffix(".txt")
    if not txt_path.exists():
        continue
    text = txt_path.read_text(encoding="utf-8").strip()
    if not text:
        continue
    # speaker_id берём из имени папки
    speaker_id = wav_path.parent.name
    samples.append({
        "path": str(wav_path),
        "text": text,
        "speaker_id": speaker_id,
    })

print(f"✅ Примеров с текстом: {len(samples)}")
dataset = Dataset.from_list(samples).cast_column("path", Audio(sampling_rate=22050))
dataset = dataset.rename_column("path", "audio")
print(dataset)

Найдено wav файлов: 108387


Парсим: 100%|██████████| 108387/108387 [00:41<00:00, 2641.68it/s]


✅ Примеров с текстом: 108387
Dataset({
    features: ['audio', 'text', 'speaker_id'],
    num_rows: 108387
})


In [ ]:
from collections import Counter

counts = Counter(s["speaker_id"] for s in samples)
top5 = counts.most_common(5)
print("Топ-5 дикторов по объёму:")
for spk, cnt in top5:
    print(f"  {spk}: {cnt} записей")

# Берём топ-1 для single-speaker TTS
TOP_SPEAKER = top5[0][0]
single_spk = dataset.filter(lambda x: x["speaker_id"] == TOP_SPEAKER)
print(f"\n✅ Отобрано: {len(single_spk)} примеров от {TOP_SPEAKER}")

Топ-5 дикторов по объёму:
  train: 100767 записей
  test: 3837 записей
  dev: 3783 записей


Filter:   0%|          | 0/108387 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [ ]:
# Смотрим на реальные имена файлов
for wav in list(EXTRACT_DIR.rglob("*.wav"))[:10]:
    print(wav.name)

2_A_2_A-64.wav
719700465_1_22338_1.wav
1246631639_1_8253_2.wav
352830655_1_34844_1.wav
733860783_1_63763_2.wav
1459541555_1_22480_2.wav
293217510_1_71487_2.wav
802338886_1_16986_1.wav
sovchilar_data_sovchilar-296.wav
609172707_1_18785_2.wav


In [ ]:
# Ищем все не-аудио файлы — там должен быть mapping
for p in EXTRACT_DIR.rglob("*"):
    if p.is_file() and p.suffix not in ('.wav', '.mp3'):
        print(p.relative_to(EXTRACT_DIR), "—", p.stat().st_size, "bytes")

Выходные данные были обрезаны до нескольких последних строк (5000).
ISSAI_USC/dev/980203197_2_83148_1.txt — 25 bytes
ISSAI_USC/dev/228981554_2_6812_2.txt — 34 bytes
ISSAI_USC/dev/258983674_1_88932_1.txt — 39 bytes
ISSAI_USC/dev/1307777999_2_23982_1.txt — 61 bytes
ISSAI_USC/dev/258983674_1_95154_1.txt — 98 bytes
ISSAI_USC/dev/105718008_2_33884_1.txt — 103 bytes
ISSAI_USC/dev/258983674_1_95161_1.txt — 53 bytes
ISSAI_USC/dev/865108248_1_87459_1.txt — 38 bytes
ISSAI_USC/dev/258983674_1_86603_1.txt — 60 bytes
ISSAI_USC/dev/913213163_2_60052_1.txt — 11 bytes
ISSAI_USC/dev/1307777999_2_23907_1.txt — 24 bytes
ISSAI_USC/dev/980203197_2_84376_1.txt — 49 bytes
ISSAI_USC/dev/671820432_2_2267_1.txt — 61 bytes
ISSAI_USC/dev/980203197_2_75933_2.txt — 19 bytes
ISSAI_USC/dev/980203197_2_84568_1.txt — 28 bytes
ISSAI_USC/dev/1217854040_2_15309_2.txt — 42 bytes
ISSAI_USC/dev/865108248_1_87461_1.txt — 42 bytes
ISSAI_USC/dev/584814067_1_22058_1.txt — 94 bytes
ISSAI_USC/dev/980203197_2_23390_1.txt — 76 bytes

In [ ]:
samples = []
wav_files = sorted(EXTRACT_DIR.rglob("*.wav"))
print(f"Найдено wav файлов: {len(wav_files)}")

for wav_path in tqdm(wav_files, desc="Парсим"):
    txt_path = wav_path.with_suffix(".txt")
    if not txt_path.exists():
        continue
    text = txt_path.read_text(encoding="utf-8").strip()
    if not text:
        continue

    # speaker_id = первый сегмент имени файла до '_'
    speaker_id = wav_path.stem.split("_")[0]
    split = wav_path.parent.name  # train / dev / test

    samples.append({
        "path": str(wav_path),
        "text": text,
        "speaker_id": speaker_id,
        "split": split,
    })

print(f"✅ Примеров: {len(samples)}")

counts = Counter(s["speaker_id"] for s in samples)
print(f"Уникальных дикторов: {len(counts)}")
print("Топ-10:", counts.most_common(10))

Найдено wav файлов: 108387


Парсим: 100%|██████████| 108387/108387 [00:06<00:00, 17371.20it/s]

✅ Примеров: 108387
Уникальных дикторов: 932
Топ-10: [('1459541555', 4897), ('1187023182', 3928), ('802338886', 3089), ('862111826', 2983), ('881144778', 2554), ('536824939', 2088), ('911079671', 1798), ('125461356', 1743), ('765131298', 1689), ('676943050', 1666)]
